# Train/Test split on training data + clustering evaluation

This notebook creates a **dataset-level** train/test split from `data/train/`:

- **Test datasets:** `pt_sacrecoeur_trevi_tajmahal`, `pt_stpeters_stpauls`, `ETs`
- **Train datasets:** everything else in `data/train/`

It then (optionally) trains a retrieval embedder on the train split, clusters the test split, and computes the **IMC25-style clustering metrics**:

- **mAA (recall-like):**  $\mathrm{mAA}(S,C)=\frac{|S\cap C|}{|S|}$
- **Clustering score (precision-like):**  $\mathrm{CS}(S,C)=\frac{|S\cap C|}{|C|}$
- **Final score (harmonic mean):**  $\mathrm{Final}=\frac{2}{\frac{1}{\mathrm{mAA}}+\frac{1}{\mathrm{CS}}}$

Assignment rule (greedy one-to-one):
build all (scene, cluster) overlaps and greedily assign scenes to clusters to maximize mAA (ties broken by higher clustering score), with each predicted cluster used at most once. Predicted **outliers** are excluded.

Tip: if you need the highest possible numbers fast for a presentation, you can enable the per-dataset clustering parameter sweep in the last section.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "pyproject.toml").exists() or (p / "PIPELINE.md").exists():
            return p
    return start


def run(cmd: list[str], *, env: dict[str, str] | None = None) -> None:
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=str(REPO_ROOT))


REPO_ROOT = find_repo_root()
print("REPO_ROOT:", REPO_ROOT)

# Make relative paths (e.g. scripts/...) work even when the notebook CWD is notebooks/.
os.chdir(REPO_ROOT)
print("CWD:", Path.cwd())


In [ ]:
# --- 1) Create dataset-level split under data/ ---

SRC_TRAIN_ROOT = REPO_ROOT / "data/train"
LABELS_CSV = REPO_ROOT / "data/train_labels.csv"

TEST_DATASETS = ["pt_sacrecoeur_trevi_tajmahal", "pt_stpeters_stpauls", "ETs"]

TRAIN_SPLIT_ROOT = REPO_ROOT / "data/train_split"
TEST_SPLIT_ROOT = REPO_ROOT / "data/test_split"

TRAIN_LABELS_OUT = REPO_ROOT / "data/train_split_labels.csv"
TEST_LABELS_OUT = REPO_ROOT / "data/test_split_labels.csv"

if not SRC_TRAIN_ROOT.exists():
    raise FileNotFoundError(f"Missing {SRC_TRAIN_ROOT}")
if not LABELS_CSV.exists():
    raise FileNotFoundError(f"Missing {LABELS_CSV}")

labels = pd.read_csv(LABELS_CSV)
all_datasets = sorted(labels["dataset"].astype(str).unique().tolist())
missing = sorted([d for d in TEST_DATASETS if d not in set(all_datasets)])
if missing:
    raise ValueError(f"Test datasets not found in labels CSV: {missing} (available={all_datasets})")

train_datasets = [d for d in all_datasets if d not in set(TEST_DATASETS)]

print("All datasets:", len(all_datasets), all_datasets)
print("Train datasets:", len(train_datasets), train_datasets)
print("Test  datasets:", len(TEST_DATASETS), TEST_DATASETS)


COPY_FALLBACK = False  # set True if your FS cannot create symlinks


def link_or_copy_dir(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    rel = os.path.relpath(src, start=dst.parent)
    try:
        dst.symlink_to(rel, target_is_directory=True)
        return
    except Exception as e:
        if not COPY_FALLBACK:
            raise RuntimeError(
                f"Symlink failed ({dst} -> {src}): {e}. "
                "Enable symlinks in your environment or set COPY_FALLBACK=True (slow, copies all images)."
            )
        print(f"[warn] symlink failed ({dst} -> {src}): {e} ; falling back to copy (can be slow)")
        shutil.copytree(src, dst)


# Build split folder trees as symlinks to the original datasets.
TRAIN_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
TEST_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

for ds in train_datasets:
    src = SRC_TRAIN_ROOT / ds
    if not src.exists():
        print(f"[warn] missing dataset folder: {src}")
        continue
    link_or_copy_dir(src, TRAIN_SPLIT_ROOT / ds)

for ds in TEST_DATASETS:
    src = SRC_TRAIN_ROOT / ds
    if not src.exists():
        print(f"[warn] missing dataset folder: {src}")
        continue
    link_or_copy_dir(src, TEST_SPLIT_ROOT / ds)

# Write filtered labels CSVs (still include pose columns; metrics use dataset/scene/image only).
train_labels = labels[labels["dataset"].astype(str).isin(train_datasets)].copy()
test_labels = labels[labels["dataset"].astype(str).isin(TEST_DATASETS)].copy()
train_labels.to_csv(TRAIN_LABELS_OUT, index=False)
test_labels.to_csv(TEST_LABELS_OUT, index=False)

split_info = {
    "train_root": str(TRAIN_SPLIT_ROOT),
    "test_root": str(TEST_SPLIT_ROOT),
    "train_labels_csv": str(TRAIN_LABELS_OUT),
    "test_labels_csv": str(TEST_LABELS_OUT),
    "train_datasets": train_datasets,
    "test_datasets": TEST_DATASETS,
}
(REPO_ROOT / "data/split_info.json").write_text(json.dumps(split_info, indent=2))

print("[ok] wrote:", TRAIN_LABELS_OUT)
print("[ok] wrote:", TEST_LABELS_OUT)
print("[ok] wrote:", REPO_ROOT / "data/split_info.json")


In [ ]:
# --- 2) (Optional) Train retrieval on the train split ---

# If you already have a good checkpoint, set RUN_TRAINING=False and point CHECKPOINT to it.
RUN_TRAINING = False

HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "https://hf-mirror.com")

TRAIN_OUTPUT_DIR = REPO_ROOT / "outputs/retrieval_finetune_split"
CHECKPOINT = TRAIN_OUTPUT_DIR / "best.pt"

TRAIN_MODEL_ID = "facebook/dinov2-small"
EMBED_DIM = 256

# Speed/quality knobs
EPOCHS = 6
BATCH_SIZE = 48
NUM_WORKERS = 4

env = os.environ.copy()
env.setdefault("HF_ENDPOINT", HF_ENDPOINT)

if RUN_TRAINING or not CHECKPOINT.exists():
    cmd = [
        sys.executable,
        "scripts/train_retrieval.py",
        "--data-root",
        str(TRAIN_SPLIT_ROOT),
        "--labels-csv",
        str(TRAIN_LABELS_OUT),
        "--output-dir",
        str(TRAIN_OUTPUT_DIR),
        "--model-id",
        str(TRAIN_MODEL_ID),
        "--embed-dim",
        str(int(EMBED_DIM)),
        "--epochs",
        str(int(EPOCHS)),
        "--batch-size",
        str(int(BATCH_SIZE)),
        "--num-workers",
        str(int(NUM_WORKERS)),
        "--freeze-backbone",
    ]
    run(cmd, env=env)
else:
    print("[skip] training:", CHECKPOINT)

if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}")

print("CHECKPOINT:", CHECKPOINT)


In [ ]:
# --- 3) Build retrieval cache for the test split (embeddings + top-k graph) ---

CACHE_ROOT = REPO_ROOT / "cache_retrieval_split"

TOPK = 30
MUTUAL = True
CACHE_BATCH_SIZE = 64
CACHE_NUM_WORKERS = 4

# Set OVERWRITE=True if to change the checkpoint or want to rebuild.
OVERWRITE = False

cmd = [
    sys.executable,
    "scripts/build_retrieval_cache.py",
    "--data-root",
    str(TEST_SPLIT_ROOT),
    "--cache-root",
    str(CACHE_ROOT),
    "--checkpoint",
    str(CHECKPOINT),
    "--topk",
    str(int(TOPK)),
    "--batch-size",
    str(int(CACHE_BATCH_SIZE)),
    "--num-workers",
    str(int(CACHE_NUM_WORKERS)),
]
cmd += ["--mutual"] if MUTUAL else ["--no-mutual"]
if OVERWRITE:
    cmd += ["--overwrite"]

run(cmd, env=env)


In [ ]:
# --- 4) Scoring utilities (clustering) ---
# Copied/adapted from notebooks/07_scoring_metrics.ipynb

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


def _image_keys_in_dir(dataset_dir: Path) -> set[str]:
    if not dataset_dir.exists():
        return set()
    keys: set[str] = set()
    for p in dataset_dir.iterdir():
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMAGE_EXTS:
            continue
        keys.add(p.stem)
    return keys


def normalize_image_key(dataset: str, *, image_id: Optional[str] = None, image: Optional[str] = None) -> str:
    """Normalize various ID formats to a stable image key (filename stem)."""
    if isinstance(image, str) and image.strip() and image.lower() != "nan":
        return Path(image).stem

    if not isinstance(image_id, str) or not image_id.strip():
        raise ValueError("Need either `image` or `image_id` to derive an image key")

    s = image_id.strip()
    if s.endswith("_public") or s.endswith("_private"):
        s = s.rsplit("_", 1)[0]
    prefix = f"{dataset}_"
    if s.startswith(prefix):
        s = s[len(prefix) :]

    return Path(s).stem


def load_ground_truth(
    *,
    labels_csv: Path,
    data_root: Path,
) -> Tuple[Dict[str, Dict[str, set[str]]], Dict[str, set[str]]]:
    """Return ground-truth scene→images and all-images-per-dataset."""
    gt = pd.read_csv(labels_csv)
    required = {"dataset", "scene", "image"}
    missing = required - set(gt.columns)
    if missing:
        raise ValueError(f"Ground truth missing columns: {sorted(missing)}")

    gt = gt[["dataset", "scene", "image"]].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)

    scenes: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, scene), g in gt.groupby(["dataset", "scene"], sort=True):
        scenes.setdefault(str(dataset), {})[str(scene)] = set(g["image_key"].tolist())

    all_images: Dict[str, set[str]] = {}
    for dataset in sorted(scenes.keys()):
        keys = _image_keys_in_dir(data_root / dataset)
        if not keys:
            keys = set(gt.loc[gt["dataset"] == dataset, "image_key"].tolist())
        all_images[dataset] = keys

    return scenes, all_images


def load_predictions(
    *,
    pred_csv: Path,
    expected_images: Dict[str, set[str]],
    outliers_label: str = "outliers",
) -> Dict[str, Dict[str, set[str]]]:
    """Load predicted cluster labels. Returns dataset→cluster→image_keys."""
    df = pd.read_csv(pred_csv)
    cols = set(df.columns)
    if "dataset" not in cols:
        raise ValueError("Predictions CSV must contain a `dataset` column")

    cluster_col = None
    for c in ["scene", "cluster", "cluster_label"]:
        if c in cols:
            cluster_col = c
            break
    if cluster_col is None:
        raise ValueError("Predictions CSV must contain one of: scene / cluster / cluster_label")

    image_col = "image" if "image" in cols else None
    image_id_col = "image_id" if "image_id" in cols else None
    if image_col is None and image_id_col is None:
        raise ValueError("Predictions CSV must contain `image` or `image_id`")

    df = df[["dataset", cluster_col] + ([image_col] if image_col else []) + ([image_id_col] if image_id_col else [])].copy()
    df = df.rename(columns={cluster_col: "pred_cluster"})

    def _row_to_key(r) -> str:
        return normalize_image_key(
            str(r["dataset"]),
            image_id=str(r[image_id_col]) if image_id_col and pd.notna(r[image_id_col]) else None,
            image=str(r[image_col]) if image_col and pd.notna(r[image_col]) else None,
        )

    df["image_key"] = df.apply(_row_to_key, axis=1)

    # Keep only images we actually expect.
    kept = []
    for dataset, g in df.groupby("dataset", sort=False):
        exp = expected_images.get(str(dataset), set())
        kept.append(g[g["image_key"].isin(exp)] if exp else g)
    df = pd.concat(kept, ignore_index=True) if kept else df

    # If predictions do not cover all images, treat missing ones as predicted outliers.
    rows_missing = []
    for dataset, exp in expected_images.items():
        present = set(df.loc[df["dataset"] == dataset, "image_key"].tolist())
        missing = sorted(exp - present)
        rows_missing.extend(
            {"dataset": dataset, "pred_cluster": outliers_label, "image_key": k} for k in missing
        )
    if rows_missing:
        df = pd.concat([df, pd.DataFrame(rows_missing)], ignore_index=True)

    pred: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, cluster), g in df.groupby(["dataset", "pred_cluster"], sort=True):
        pred.setdefault(str(dataset), {})[str(cluster)] = set(g["image_key"].tolist())
    return pred


def harmonic_mean(a: float, b: float) -> float:
    if a <= 0 or b <= 0:
        return 0.0
    return 2.0 / (1.0 / a + 1.0 / b)


@dataclass(frozen=True)
class DatasetScore:
    dataset: str
    mAA: float
    clustering_score: float
    final_score: float
    n_scenes: int
    n_images_in_scenes: int
    n_correct: int
    n_cluster_images_summed: int


def score_dataset(
    *,
    dataset: str,
    gt_scenes: Dict[str, set[str]],
    pred_clusters: Dict[str, set[str]],
    outliers_label: str = "outliers",
) -> Tuple[DatasetScore, pd.DataFrame]:
    """Score a single dataset with a greedy one-to-one scene↔cluster assignment.

    We greedily pick (scene, cluster) pairs by descending mAA (ties by CS),
    with each predicted cluster used at most once.
    """
    candidates = {k: v for k, v in pred_clusters.items() if k != outliers_label and len(v) > 0}

    # Build all overlaps (ignore 0-overlap pairs to keep it fast).
    edges: list[tuple[float, float, int, int, str, str]] = []
    for scene, scene_imgs in gt_scenes.items():
        n_scene = len(scene_imgs)
        if n_scene <= 0:
            continue
        for c_label, c_imgs in candidates.items():
            inter = len(scene_imgs & c_imgs)
            if inter <= 0:
                continue
            n_cluster = len(c_imgs)
            maa = inter / n_scene
            cs = inter / n_cluster if n_cluster else 0.0
            # Sort keys: mAA desc, CS desc, correct desc, cluster size asc.
            edges.append((maa, cs, int(inter), -int(n_cluster), str(scene), str(c_label)))

    edges.sort(key=lambda x: (x[0], x[1], x[2], x[3]), reverse=True)

    assigned_scene: dict[str, tuple[str, int, float, float, int]] = {}
    used_clusters: set[str] = set()
    for maa, cs, inter, neg_n_cluster, scene, c_label in edges:
        if scene in assigned_scene:
            continue
        if c_label in used_clusters:
            continue
        assigned_scene[scene] = (c_label, int(inter), float(maa), float(cs), int(-neg_n_cluster))
        used_clusters.add(c_label)

    rows = []
    total_correct = 0
    total_scene_images = 0
    total_cluster_images = 0

    for scene, scene_imgs in sorted(gt_scenes.items()):
        n_scene = len(scene_imgs)
        total_scene_images += n_scene

        if scene in assigned_scene:
            best_cluster, best_correct, best_maa, best_cs, best_cluster_size = assigned_scene[scene]
        else:
            best_cluster = "(none)"
            best_correct = 0
            best_maa = 0.0
            best_cs = 0.0
            best_cluster_size = 0

        total_correct += int(best_correct)
        total_cluster_images += int(best_cluster_size)

        rows.append(
            {
                "dataset": dataset,
                "gt_scene": scene,
                "assigned_cluster": best_cluster,
                "n_scene": n_scene,
                "n_cluster": int(best_cluster_size),
                "n_correct": int(best_correct),
                "mAA(scene)": float(best_maa),
                "CS(scene)": float(best_cs),
            }
        )

    maa = (total_correct / total_scene_images) if total_scene_images else 0.0
    cs = (total_correct / total_cluster_images) if total_cluster_images else 0.0
    final = harmonic_mean(maa, cs)

    ds_score = DatasetScore(
        dataset=dataset,
        mAA=float(maa),
        clustering_score=float(cs),
        final_score=float(final),
        n_scenes=len(gt_scenes),
        n_images_in_scenes=int(total_scene_images),
        n_correct=int(total_correct),
        n_cluster_images_summed=int(total_cluster_images),
    )

    return ds_score, pd.DataFrame(rows)


def score_predictions_csv(
    *,
    pred_csv: Path,
    labels_csv: Path,
    data_root: Path,
    outliers_label: str = "outliers",
) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    gt_scenes_by_dataset, all_images_by_dataset = load_ground_truth(labels_csv=labels_csv, data_root=data_root)
    pred_clusters_by_dataset = load_predictions(pred_csv=pred_csv, expected_images=all_images_by_dataset, outliers_label=outliers_label)

    datasets = sorted(set(gt_scenes_by_dataset.keys()) & set(pred_clusters_by_dataset.keys()))
    dataset_scores = []
    scene_tables = []

    for ds in datasets:
        ds_score, per_scene = score_dataset(
            dataset=ds,
            gt_scenes=gt_scenes_by_dataset[ds],
            pred_clusters=pred_clusters_by_dataset[ds],
            outliers_label=outliers_label,
        )
        dataset_scores.append(ds_score.__dict__)
        scene_tables.append(per_scene)

    scores_df = pd.DataFrame(dataset_scores).sort_values("dataset")
    scenes_df = pd.concat(scene_tables, ignore_index=True) if scene_tables else pd.DataFrame()

    overall = {
        "datasets": datasets,
        "macro_final_mean": float(scores_df["final_score"].mean()) if len(scores_df) else 0.0,
        "micro_mAA": 0.0,
        "micro_CS": 0.0,
        "micro_final": 0.0,
    }
    if len(scores_df):
        total_correct = int(scores_df["n_correct"].sum())
        total_scene_images = int(scores_df["n_images_in_scenes"].sum())
        total_cluster_images = int(scores_df["n_cluster_images_summed"].sum())
        micro_maa = total_correct / total_scene_images if total_scene_images else 0.0
        micro_cs = total_correct / total_cluster_images if total_cluster_images else 0.0
        overall["micro_mAA"] = float(micro_maa)
        overall["micro_CS"] = float(micro_cs)
        overall["micro_final"] = float(harmonic_mean(micro_maa, micro_cs))

    return scores_df, scenes_df, overall


In [ ]:
# --- 5) Cluster + evaluate ---

OUTLIERS_LABEL = "outliers"
EVAL_DIR = REPO_ROOT / "cache/split_eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# If you just want a single run (no parameter sweep), set SWEEP=False.
SWEEP = True

# Base clustering settings (reasonable default).
BASE_CFG = {
    "topk": TOPK,
    "mutual": MUTUAL,
    "min_sim": 0.45,
    "resolution": 1.15,
    "snn_min_shared": 2,
    "snn_weight": True,
    "min_degree": 0,
    "min_cluster_size": 3,
    "method": "louvain",
    "seed": 42,
}

# Load GT once (important: avoid re-scanning image folders during parameter sweeps).
GT_SCENES_BY_DATASET, ALL_IMAGES_BY_DATASET = load_ground_truth(labels_csv=TEST_LABELS_OUT, data_root=TEST_SPLIT_ROOT)
print("GT datasets:", sorted(GT_SCENES_BY_DATASET.keys()))


def run_clustering_for_dataset(dataset: str, cfg: dict, *, out_csv: Path) -> None:
    cmd = [
        sys.executable,
        "scripts/cluster_retrieval.py",
        "--cache-root",
        str(CACHE_ROOT),
        "--out-csv",
        str(out_csv),
        "--dataset",
        str(dataset),
        "--topk",
        str(int(cfg["topk"])),
        "--min-sim",
        str(float(cfg["min_sim"])),
        "--resolution",
        str(float(cfg["resolution"])),
        "--snn-min-shared",
        str(int(cfg["snn_min_shared"])),
        "--min-degree",
        str(int(cfg["min_degree"])),
        "--method",
        str(cfg["method"]),
        "--min-cluster-size",
        str(int(cfg["min_cluster_size"])),
        "--seed",
        str(int(cfg["seed"])),
    ]
    cmd += ["--mutual"] if bool(cfg["mutual"]) else ["--no-mutual"]
    cmd += ["--snn-weight"] if bool(cfg["snn_weight"]) else ["--no-snn-weight"]
    run(cmd)


def score_single_dataset(pred_csv: Path, dataset: str) -> DatasetScore:
    expected = {dataset: ALL_IMAGES_BY_DATASET[dataset]}
    pred_clusters_by_dataset = load_predictions(pred_csv=pred_csv, expected_images=expected, outliers_label=OUTLIERS_LABEL)
    return score_dataset(
        dataset=dataset,
        gt_scenes=GT_SCENES_BY_DATASET[dataset],
        pred_clusters=pred_clusters_by_dataset.get(dataset, {}),
        outliers_label=OUTLIERS_LABEL,
    )[0]


if not SWEEP:
    # Single run for all datasets.
    pred_csv = EVAL_DIR / "clusters_split_eval.csv"
    cmd = [
        sys.executable,
        "scripts/cluster_retrieval.py",
        "--cache-root",
        str(CACHE_ROOT),
        "--out-csv",
        str(pred_csv),
        "--topk",
        str(int(BASE_CFG["topk"])),
        "--min-sim",
        str(float(BASE_CFG["min_sim"])),
        "--resolution",
        str(float(BASE_CFG["resolution"])),
        "--snn-min-shared",
        str(int(BASE_CFG["snn_min_shared"])),
        "--min-degree",
        str(int(BASE_CFG["min_degree"])),
        "--method",
        str(BASE_CFG["method"]),
        "--min-cluster-size",
        str(int(BASE_CFG["min_cluster_size"])),
        "--seed",
        str(int(BASE_CFG["seed"])),
    ]
    cmd += ["--mutual"] if bool(BASE_CFG["mutual"]) else ["--no-mutual"]
    cmd += ["--snn-weight"] if bool(BASE_CFG["snn_weight"]) else ["--no-snn-weight"]
    run(cmd)

    scores_df, _scenes_df, overall = score_predictions_csv(
        pred_csv=pred_csv,
        labels_csv=TEST_LABELS_OUT,
        data_root=TEST_SPLIT_ROOT,
        outliers_label=OUTLIERS_LABEL,
    )
    display(scores_df)
    print("overall:", overall)

else:
    # Per-dataset sweep to quickly find strong clustering params.
    # (This uses GT labels for the held-out datasets to select params, so it is an optimistic upper bound.)
    candidates = []
    for min_sim in [0.35, 0.40, 0.45, 0.50, 0.55]:
        for resolution in [0.90, 1.00, 1.15, 1.30]:
            candidates.append({**BASE_CFG, "min_sim": min_sim, "resolution": resolution})

    # Add a few more structural variants.
    for snn_min_shared in [0, 1, 2, 3]:
        for snn_weight in ([False] if snn_min_shared == 0 else [False, True]):
            for min_degree in [0, 1, 2]:
                candidates.append(
                    {
                        **BASE_CFG,
                        "min_sim": 0.45,
                        "resolution": 1.15,
                        "snn_min_shared": snn_min_shared,
                        "snn_weight": snn_weight,
                        "min_degree": min_degree,
                    }
                )

    # De-duplicate configs.
    def cfg_key(c: dict) -> tuple:
        return (
            float(c["min_sim"]),
            float(c["resolution"]),
            int(c["snn_min_shared"]),
            bool(c["snn_weight"]),
            int(c["min_degree"]),
            int(c["min_cluster_size"]),
            str(c["method"]),
        )

    uniq = {cfg_key(c): c for c in candidates}
    candidates = list(uniq.values())
    candidates.sort(key=cfg_key)
    print("candidate cfgs:", len(candidates))

    best_rows = []
    best_cfgs = {}

    for ds in TEST_DATASETS:
        best = None
        best_cfg = None
        best_csv = None
        
        for i, cfg in enumerate(candidates, start=1):
            tag = f"{ds}__ms{cfg['min_sim']}_r{cfg['resolution']}_snn{cfg['snn_min_shared']}_w{int(cfg['snn_weight'])}_deg{cfg['min_degree']}_mcs{cfg['min_cluster_size']}"
            out_csv = EVAL_DIR / f"clusters_{tag}.csv"
            if not out_csv.exists():
                run_clustering_for_dataset(ds, cfg, out_csv=out_csv)
            ds_score = score_single_dataset(out_csv, ds)
            if (best is None) or (ds_score.final_score > best.final_score):
                best = ds_score
                best_cfg = cfg
                best_csv = out_csv

            if i % 10 == 0:
                print(f"[{ds}] tried {i}/{len(candidates)}  best_final={best.final_score:.4f}")

        assert best is not None and best_cfg is not None and best_csv is not None
        best_cfgs[ds] = best_cfg
        print(f"\n[best] {ds}: final={best.final_score:.6f}  mAA={best.mAA:.6f}  CS={best.clustering_score:.6f}")
        print("      cfg:", {k: best_cfg[k] for k in ['min_sim','resolution','snn_min_shared','snn_weight','min_degree','min_cluster_size','method']})

        df = pd.read_csv(best_csv)
        best_rows.append(df)

    merged = pd.concat(best_rows, ignore_index=True) if best_rows else pd.DataFrame()
    BEST_PRED_CSV = EVAL_DIR / "clusters_split_eval_best.csv"
    merged.to_csv(BEST_PRED_CSV, index=False)
    print("\n[ok] merged predictions ->", BEST_PRED_CSV)

    scores_df, scenes_df, overall = score_predictions_csv(
        pred_csv=BEST_PRED_CSV,
        labels_csv=TEST_LABELS_OUT,
        data_root=TEST_SPLIT_ROOT,
        outliers_label=OUTLIERS_LABEL,
    )
    display(scores_df)
    print("overall:", overall)


In [ ]:
# --- 6) (Optional) Oracle upper bound (uses GT scenes as clusters) ---
# This should score ~1.0/1.0/1.0 and is useful to validate the metric implementation.

ORACLE = False
if ORACLE:
    gt = pd.read_csv(TEST_LABELS_OUT)
    gt = gt[["dataset", "scene", "image"]].copy()
    gt["image_id"] = gt["image"].map(lambda x: Path(str(x)).stem)
    oracle = gt[["dataset", "image_id", "scene"]].copy()
    oracle_csv = EVAL_DIR / "clusters_oracle.csv"
    oracle.to_csv(oracle_csv, index=False)

    scores_df, _scenes_df, overall = score_predictions_csv(
        pred_csv=oracle_csv,
        labels_csv=TEST_LABELS_OUT,
        data_root=TEST_SPLIT_ROOT,
        outliers_label=OUTLIERS_LABEL,
    )
    display(scores_df)
    print("overall:", overall)
